# Sea Urchin Gastrulation Long-Horizon Simulation (100,000 Iterations)

This notebook runs the sea urchin gastrulation simulation in EmbryoMaker Flow for up to 100,000 iterations, stores the complete trajectory across all iterations, and renders an animated cross-section video in Google Colab.

### Key features
- **Long horizon:** Simulates up to 100,000 iterations in chunked batches.
- **Checkpoint persistence:** Checkpoints stream directly to Google Drive so progress is never lost.
- **Automatic resume:** If the runtime disconnects, re-running the cell picks up from the latest checkpoint.
- **Full trajectory storage:** Exports both structured `.embtraj` (for EmbryoMaker Studio) and `.npz` (for NumPy and scientific analysis).

In [ ]:
# Cell 1: Mount Google Drive, set workspace, and fetch simulation binary
from pathlib import Path
import os, shutil, subprocess, tarfile, urllib.request

WORK_DIR = Path('/content/embryomaker_sea_urchin')
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Mount Google Drive for persistent storage
DRIVE_DIR = None
DRIVE_CHECKPOINTS = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = Path('/content/drive/MyDrive/EmbryoMaker_Sea_Urchin')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_CHECKPOINTS = DRIVE_DIR / 'checkpoints'
    DRIVE_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
    print(f'Google Drive mounted. Storage path: {DRIVE_DIR}')
except Exception as e:
    print(f'Running locally or without Google Drive: {e}')

ARCHIVE_URL = 'https://godofecht.github.io/embryo-maker-flow/downloads/EmbryoMaker-Flow-linux-x86_64-v0.1.0.tar.gz'
ARCHIVE_PATH = WORK_DIR / 'embryomaker-linux.tar.gz'

if not ARCHIVE_PATH.exists():
    print('Downloading EmbryoMaker Flow Linux binary...')
    urllib.request.urlretrieve(ARCHIVE_URL, ARCHIVE_PATH)

with tarfile.open(ARCHIVE_PATH, 'r:gz') as archive:
    root = WORK_DIR.resolve()
    members = []
    for member in archive.getmembers():
        target = (WORK_DIR / member.name).resolve()
        if target != root and root not in target.parents:
            raise ValueError(f'Unsafe archive path: {member.name}')
        if '/._' not in member.name and not Path(member.name).name.startswith('._'):
            members.append(member)
    archive.extractall(WORK_DIR, members=members, filter='data')

PROJECT = WORK_DIR / 'EmbryoMaker-Flow'
BINARY = PROJECT / 'EmbryoMaker'
INPUT_FILE = PROJECT / 'reference/examples/sea_urchin_gastrulation/sea_urchin.txt'

assert BINARY.is_file(), 'Simulator binary was not found in release archive.'
assert INPUT_FILE.is_file(), 'Sea urchin input file was not found.'
BINARY.chmod(0o755)
print('Simulator binary ready:', BINARY)
print('Sea urchin input ready:', INPUT_FILE)

In [ ]:
# Cell 2: Run Sea Urchin Gastrulation up to 100,000 iterations (with automatic checkpointing and resume)
import time

FRAMES_DIR = WORK_DIR / 'frames'
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

# Restore existing checkpoints from Google Drive if present
if DRIVE_CHECKPOINTS and DRIVE_CHECKPOINTS.exists():
    drive_files = list(DRIVE_CHECKPOINTS.glob('urchin_*.dat'))
    if drive_files:
        print(f'Syncing {len(drive_files)} existing checkpoints from Google Drive...')
        for f in drive_files:
            dest = FRAMES_DIR / f.name
            if not dest.exists():
                shutil.copy2(f, dest)

TARGET_ITERATIONS = 100_000
INTERVAL = 500          # Save snapshot every 500 iterations (200 checkpoints total)
CHUNK_SNAPSHOTS = 10    # 5,000 iterations per chunk before syncing to Drive
PREFIX = FRAMES_DIR / 'urchin'

def get_latest_checkpoint(directory: Path):
    existing = sorted(
        directory.glob('urchin_*.dat'),
        key=lambda p: int(p.stem.split('_')[1]) if p.stem.split('_')[1].isdigit() else 0
    )
    if not existing:
        return None, 0
    latest = existing[-1]
    iteration = int(latest.stem.split('_')[1])
    return latest, iteration

latest_file, current_iter = get_latest_checkpoint(FRAMES_DIR)
print(f'Starting status: Current iteration = {current_iter}, Target iteration = {TARGET_ITERATIONS}')

total_start_time = time.time()

while current_iter < TARGET_ITERATIONS:
    remaining_iters = TARGET_ITERATIONS - current_iter
    remaining_snapshots = (remaining_iters + INTERVAL - 1) // INTERVAL
    chunk_count = min(CHUNK_SNAPSHOTS, remaining_snapshots)
    chunk_iters = chunk_count * INTERVAL
    next_target = current_iter + chunk_iters
    
    print(f'\n--- Simulating iterations {current_iter} -> {next_target} ({chunk_count} checkpoints) ---')
    input_source = latest_file if latest_file else INPUT_FILE
    
    cmd = [str(BINARY), str(input_source), '1', str(INTERVAL), str(chunk_count), str(PREFIX)]
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    t1 = time.time()
    
    if proc.returncode != 0:
        print('STDERR:', proc.stderr)
        raise RuntimeError(f'Simulation stopped at iteration {current_iter}: exit code {proc.returncode}')
        
    latest_file, current_iter = get_latest_checkpoint(FRAMES_DIR)
    elapsed = t1 - t0
    rate = chunk_iters / max(0.001, elapsed)
    print(f'Chunk finished in {elapsed:.1f}s ({rate:.1f} it/s). Reached iteration {current_iter}.')
    
    # Backup checkpoints to Google Drive
    if DRIVE_CHECKPOINTS:
        for f in FRAMES_DIR.glob('urchin_*.dat'):
            drive_dest = DRIVE_CHECKPOINTS / f.name
            if not drive_dest.exists():
                shutil.copy2(f, drive_dest)
        print(f'Drive sync: {len(list(DRIVE_CHECKPOINTS.glob("urchin_*.dat")))} checkpoints secured.')

total_elapsed = time.time() - total_start_time
dat_files = sorted(FRAMES_DIR.glob('urchin_*.dat'), key=lambda p: int(p.stem.split('_')[1]))
print(f'\nRun finished. Reached iteration {current_iter} with {len(dat_files)} checkpoints.')

In [ ]:
# Cell 3: Snapshot parsing routine for analysis and video rendering
import numpy as np

def parse_snapshot(path: Path):
    lines = path.read_text(encoding='utf-8', errors='replace').splitlines()
    
    try:
        node_sec = lines.index('node properties')
    except ValueError:
        node_sec = lines.index('node parameters')
        
    cursor = node_sec + 1
    while cursor < len(lines) and not lines[cursor].strip():
        cursor += 1
    if cursor < len(lines) and lines[cursor].strip() == 'node parameters':
        cursor += 1
        
    nodes = []
    while cursor < len(lines) and lines[cursor].strip():
        fields = lines[cursor].split()
        x, y, z = float(fields[0]), float(fields[1]), float(fields[2])
        tipus = int(fields[28]) if len(fields) > 28 else 1
        cell = int(fields[29]) if len(fields) > 29 else 1
        radius = float(fields[5]) if len(fields) > 5 else 0.2
        nodes.append((x, y, z, tipus, cell, radius))
        cursor += 1
        
    gene1 = [0.0] * len(nodes)
    try:
        g_sec = lines.index('G matrix: gene expression')
        cursor = g_sec + 1
        while cursor < len(lines) and lines[cursor].strip() != 'node gene values':
            cursor += 1
        cursor += 1
        while cursor < len(lines) and lines[cursor].strip():
            fields = lines[cursor].split()
            node_idx = int(fields[0]) - 1
            if 0 <= node_idx < len(nodes) and len(fields) > 1:
                gene1[node_idx] = float(fields[1])
            cursor += 1
    except ValueError:
        pass
        
    coords = np.array([(n[0], n[1], n[2]) for n in nodes], dtype=np.float64)
    types = np.array([n[3] for n in nodes], dtype=np.int32)
    cells = np.array([n[4] for n in nodes], dtype=np.int32)
    radii = np.array([n[5] for n in nodes], dtype=np.float64)
    g1 = np.array(gene1, dtype=np.float64)
    return coords, types, cells, radii, g1

if dat_files:
    c, t, ce, r, g1 = parse_snapshot(dat_files[-1])
    print(f'Latest checkpoint: {dat_files[-1].name}')
    print(f'Node count: {len(c)}')
    print(f'Cell types present: {np.unique(t)}')

In [ ]:
# Cell 4: Render animated cross-section video of the full trajectory
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
from matplotlib.colors import to_rgba
from IPython.display import Video, display

VIDEO_PATH = WORK_DIR / 'sea_urchin_gastrulation.mp4'
writer = FFMpegWriter(fps=12, metadata=dict(title='Sea Urchin Gastrulation in EmbryoMaker Flow'), bitrate=2400)

fig = plt.figure(figsize=(9, 9), facecolor='#0e1117')
ax = fig.add_subplot(111, projection='3d', facecolor='#0e1117')

# Subsample frames if snapshot count is large (e.g. up to 100 frames for smooth video)
max_video_frames = 100
step = max(1, len(dat_files) // max_video_frames)
selected_files = dat_files[::step]
if dat_files and dat_files[-1] not in selected_files:
    selected_files.append(dat_files[-1])

print(f'Rendering {len(selected_files)} frames into MP4 video...')
with writer.saving(fig, str(VIDEO_PATH), dpi=140):
    for frame_idx, dat_file in enumerate(selected_files):
        ax.clear()
        ax.set_facecolor('#0e1117')
        ax.grid(False)
        ax.axis('off')
        
        coords, types, cells, radii, g1 = parse_snapshot(dat_file)
        iteration_num = int(dat_file.stem.split('_')[1])
        
        # Cross-section slice along Y-axis to reveal blastocoel cavity
        mask = coords[:, 1] >= -0.15
        c_slice = coords[mask]
        t_slice = types[mask]
        g_slice = g1[mask]
        
        rgba_colors = []
        sizes = []
        
        for typ, g in zip(t_slice, g_slice):
            if typ == 4:  # ECM envelope
                rgba_colors.append(to_rgba('#4a5568', 0.20))
                sizes.append(10)
            elif typ == 3:  # Mesenchyme cells
                rgba_colors.append(to_rgba('#d53f8c', 0.95))
                sizes.append(26)
            elif g > 0.05:  # Vegetal pole expression
                rgba_colors.append(to_rgba('#ff5722', 0.95))
                sizes.append(28)
            elif typ == 1:  # Apical epithelium
                rgba_colors.append(to_rgba('#f6ad55', 0.90))
                sizes.append(22)
            else:  # Basal epithelium
                rgba_colors.append(to_rgba('#319795', 0.90))
                sizes.append(22)
                
        ax.scatter(
            c_slice[:, 0], c_slice[:, 1], c_slice[:, 2],
            c=rgba_colors, s=sizes, depthshade=True
        )
        
        ax.set_xlim(-6.5, 6.5)
        ax.set_ylim(-6.5, 6.5)
        ax.set_zlim(-6.5, 6.5)
        ax.view_init(elev=18, azim=135)
        
        ax.set_title(
            f'Sea Urchin Gastrulation: Iteration {iteration_num:,}\nVegetal Invagination and Archenteron Formation',
            color='#e2e8f0', fontsize=13, pad=-20
        )
        
        writer.grab_frame()
        if (frame_idx + 1) % 10 == 0 or frame_idx == len(selected_files) - 1:
            print(f'Rendered frame {frame_idx + 1}/{len(selected_files)}')

plt.close(fig)
print('Video compilation complete:', VIDEO_PATH)
display(Video(str(VIDEO_PATH), embed=True, html_attributes='controls autoplay loop style="max-width: 640px; border-radius: 8px;"'))

In [ ]:
# Cell 5: Save structured trajectory (.npz and .embtraj) and raw snapshots to Google Drive
import gzip, json, tarfile
import numpy as np

print('Packaging full trajectory across all checkpoints...')
all_coords, all_types, all_cells, all_radii, all_g1, all_iterations = [], [], [], [], [], []
embtraj_frames = []

for f in dat_files:
    it_num = int(f.stem.split('_')[1])
    c, t, ce, r, g = parse_snapshot(f)
    all_coords.append(c)
    all_types.append(t)
    all_cells.append(ce)
    all_radii.append(r)
    all_g1.append(g)
    all_iterations.append(it_num)
    
    # Frame for web-ready .embtraj format
    nodes_data = []
    for idx in range(len(c)):
        nodes_data.append({
            'id': idx + 1,
            'x': float(c[idx, 0]), 'y': float(c[idx, 1]), 'z': float(c[idx, 2]),
            'radius': float(r[idx]), 'eqd': float(r[idx] * 0.5),
            'type': int(t[idx]), 'cell': int(ce[idx]),
            'genes': [float(g[idx])]
        })
    embtraj_frames.append({
        'iteration': it_num,
        'count': len(c),
        'nodes': nodes_data,
        'cells': [],
        'edges': []
    })

# Build padded arrays for NPZ container
max_nodes = max(len(c) for c in all_coords)
n_frames = len(all_coords)

padded_coords = np.full((n_frames, max_nodes, 3), np.nan, dtype=np.float64)
padded_types = np.full((n_frames, max_nodes), -1, dtype=np.int32)
padded_cells = np.full((n_frames, max_nodes), -1, dtype=np.int32)
padded_radii = np.full((n_frames, max_nodes), np.nan, dtype=np.float64)
padded_g1 = np.full((n_frames, max_nodes), np.nan, dtype=np.float64)
node_counts = np.array([len(c) for c in all_coords], dtype=np.int32)

for i in range(n_frames):
    k = len(all_coords[i])
    padded_coords[i, :k] = all_coords[i]
    padded_types[i, :k] = all_types[i]
    padded_cells[i, :k] = all_cells[i]
    padded_radii[i, :k] = all_radii[i]
    padded_g1[i, :k] = all_g1[i]

npz_path = WORK_DIR / 'sea_urchin_trajectory.npz'
np.savez_compressed(
    npz_path,
    iterations=np.array(all_iterations, dtype=np.int32),
    node_counts=node_counts,
    coords=padded_coords,
    types=padded_types,
    cells=padded_cells,
    radii=padded_radii,
    gene1=padded_g1
)

# Save web-ready .embtraj.gz format
embtraj_path = WORK_DIR / 'sea_urchin_trajectory.embtraj.gz'
traj_dict = {
    'format': 'embryomaker-trajectory',
    'version': 1,
    'experiment': 'Sea Urchin Gastrulation',
    'totalIterations': all_iterations[-1] if all_iterations else 0,
    'minIteration': all_iterations[0] if all_iterations else 0,
    'maxIteration': all_iterations[-1] if all_iterations else 0,
    'frameCount': len(embtraj_frames),
    'iterations': all_iterations,
    'frames': embtraj_frames
}
with gzip.open(embtraj_path, 'wt', encoding='utf-8') as f:
    json.dump(traj_dict, f, separators=(',', ':'))

# Package raw checkpoints into tar.gz
tar_path = WORK_DIR / 'sea_urchin_raw_snapshots.tar.gz'
with tarfile.open(tar_path, 'w:gz') as tar:
    for f in dat_files:
        tar.add(f, arcname=f.name)

print(f'Packaged NumPy trajectory (NPZ): {npz_path} ({npz_path.stat().st_size / 1024 / 1024:.2f} MB)')
print(f'Packaged EmbryoMaker Studio trajectory (.embtraj.gz): {embtraj_path} ({embtraj_path.stat().st_size / 1024 / 1024:.2f} MB)')
print(f'Packaged raw snapshots (.tar.gz): {tar_path} ({tar_path.stat().st_size / 1024 / 1024:.2f} MB)')

if DRIVE_DIR:
    shutil.copy2(VIDEO_PATH, DRIVE_DIR / VIDEO_PATH.name)
    shutil.copy2(npz_path, DRIVE_DIR / npz_path.name)
    shutil.copy2(embtraj_path, DRIVE_DIR / embtraj_path.name)
    shutil.copy2(tar_path, DRIVE_DIR / tar_path.name)
    print('\nAll artifacts safely synchronized to Google Drive:')
    print(f' - Video: {DRIVE_DIR / VIDEO_PATH.name}')
    print(f' - NumPy Trajectory: {DRIVE_DIR / npz_path.name}')
    print(f' - Studio Trajectory: {DRIVE_DIR / embtraj_path.name}')
    print(f' - Raw Snapshots: {DRIVE_DIR / tar_path.name}')
else:
    print('\nFiles available locally under:', WORK_DIR)